# KiwKiw / RVC Training (versi bersih)

**Sebelum mulai:** Runtime → Change runtime type → **T4 GPU** (mengganti tipe runtime me-reset sesi, jadi lakukan ini PERTAMA).

**Urutan:**
1. Jalankan cell 0 dan 1 (cell 1 hanya sekali per sesi baru).
2. Setelah cell 1 selesai: **Runtime → Restart session** (bukan "Disconnect and delete runtime").
3. Setelah restart, **lewati cell 1**. Mulai dari cell 2 sampai cell 9.

Kalau memakai "Run all" setelah restart, cell 1 akan otomatis skip karena sudah ada penanda `.installed`.

In [ ]:
#@title 0. Mount Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#@title 1. Install (cukup SEKALI per sesi baru)
zip_path = "/content/drive/MyDrive/assets/KiwKiw.zip" #@param {type:"string"}
import os

if os.system("nvidia-smi -L > /dev/null 2>&1") != 0:
    raise SystemExit("GPU tidak terdeteksi. Runtime > Change runtime type > T4 GPU, lalu mulai lagi dari cell 0.")

if os.path.exists("/content/KiwKiw/.installed"):
    print("Sudah terinstall di sesi ini, SKIP.")
else:
    !unzip -n -q "{zip_path}" -d /content
    %cd /content/KiwKiw
    !pip install -q pip==23.3.1
    !sed -i 's/onnxruntime-gpu.*/onnxruntime-gpu/g' requirments_cu128_py312.txt
    # buang dependency WebUI (gradio dkk) supaya tidak merusak package Colab
    !grep -vE '^(gradio|altair|anyio|fastapi|httpx|markdown-it-py|mdit-py-plugins|pydantic|pyparsing|starlette|uvicorn|websockets)' requirments_cu128_py312.txt > req_train.txt
    !pip install -r req_train.txt
    # TensorFlow & jax bawaan Colab bentrok dengan numpy 1.26 dan tidak dibutuhkan
    !pip uninstall -y tensorflow tensorflow-cpu tf-keras keras jax jaxlib

    os.makedirs("assets/weights", exist_ok=True)
    os.makedirs("assets/indices", exist_ok=True)

    # hubert_base (format HuggingFace) + logs/mute dari repo resmi
    dl = [
        "from huggingface_hub import snapshot_download, hf_hub_download",
        "import zipfile",
        "snapshot_download('lj1995/VoiceConversionWebUI', allow_patterns=['hubert_base/*'], local_dir='assets')",
        "p = hf_hub_download('lj1995/VoiceConversionWebUI', 'mute.zip', local_dir='.model-downloads')",
        "zipfile.ZipFile(p).extractall('logs')",
    ]
    open("/content/dl_models.py", "w").write("\n".join(dl) + "\n")
    !python /content/dl_models.py

    need = [
        "assets/hubert_base/pytorch_model.bin",
        "assets/hubert_base/config.json",
        "assets/hubert_base/preprocessor_config.json",
        "assets/rmvpe/rmvpe.pt",
        "assets/pretrained_v2/f0G48k.pth",
        "logs/mute/0_gt_wavs",
    ]
    missing = [p for p in need if not os.path.exists(p)]
    if missing:
        raise RuntimeError("File berikut tidak ditemukan: %s" % missing)

    open(".installed", "w").close()
    print("\nINSTALL SELESAI.")
    print("Sekarang: Runtime > Restart session, lalu lanjut ke cell 2 (JANGAN ulang cell ini).")

In [ ]:
#@title 2. Cek environment (jalankan setelah restart)
import os
if not os.path.exists("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount('/content/drive')

if not os.path.exists("/content/KiwKiw/.installed"):
    raise SystemExit("Belum terinstall. Jalankan cell 1 dulu.")

os.chdir("/content/KiwKiw")
import numpy as np
if not np.__version__.startswith("1.26"):
    raise SystemExit("numpy %s terdeteksi. Lakukan Runtime > Restart session, lalu jalankan cell ini lagi (skip cell 1)." % np.__version__)

# berlaku untuk semua proses yang dijalankan dari notebook
os.environ["PYTHONPATH"] = "/content/KiwKiw"
os.environ["USE_TF"] = "0"
os.environ["USE_JAX"] = "0"
os.environ["USE_FLAX"] = "0"
print("OK. numpy", np.__version__, "| siap lanjut ke cell 3.")

In [ ]:
#@title 3. Pengaturan model
import os

model_name = "dusk" #@param {type:"string"}
dataset = "/content/drive/MyDrive/dataset/dusk" #@param {type:"string"}
sample_rate = "48000" #@param ["40000", "48000"]
version = "v2" #@param ["v1", "v2"]
f0method = "rmvpe_gpu" #@param ["pm", "dio", "harvest", "rmvpe", "rmvpe_gpu"]
save_frequency = 50 #@param {type:"slider", min:1, max:50, step:1}
epoch = 300 #@param {type:"integer"}
batch_size = 7 #@param {type:"integer"}
cache_gpu = True #@param {type:"boolean"}

exp_dir = model_name
ksample_rate = "40k" if sample_rate == "40000" else "48k"
now_dir = "/content/KiwKiw"
exp_path = f"{now_dir}/logs/{exp_dir}"
os.chdir(now_dir)
os.makedirs(exp_path, exist_ok=True)

if not os.path.isdir(dataset):
    raise RuntimeError(f"Folder dataset tidak ditemukan: {dataset}")
audio_ext = (".wav", ".mp3", ".flac", ".ogg", ".m4a")
n_audio = sum(1 for f in os.listdir(dataset) if f.lower().endswith(audio_ext))
if n_audio == 0:
    raise RuntimeError(f"Tidak ada file audio di {dataset}")
print(f"Model: {exp_dir} | {ksample_rate} | {version} | {n_audio} file audio | {epoch} epoch | batch {batch_size}")

In [ ]:
#@title 4. Process Data
import os
os.chdir(now_dir)
command = f"python -m train.preprocess '{dataset}' {sample_rate} 2 '{exp_path}' False 3.0"
print(command)
!{command}

gt = f"{exp_path}/0_gt_wavs"
n = len(os.listdir(gt)) if os.path.isdir(gt) else 0
if n == 0:
    raise RuntimeError("Preprocess gagal: 0_gt_wavs kosong. Baca error di atas.")
print(f"OK: {n} potongan audio.")

In [ ]:
#@title 5. Feature Extraction (F0 + HuBERT)
import os
os.chdir(now_dir)

if f0method != "rmvpe_gpu":
    command = f"python -m train.dataset.extract_f0 cpu '{exp_path}' 2 '{f0method}'"
else:
    command = f"python -m train.dataset.extract_f0 cuda 1 0 0 '{exp_path}' True"
print(command)
!{command}

command = f"python -m train.dataset.extract_hubert_feature cuda:0 1 0 0 '{exp_path}' '{version}' False"
print(command)
!{command}

fea_dim = 256 if version == "v1" else 768
for sub in ["2a_f0", "2b-f0nsf", f"3_feature{fea_dim}"]:
    d = f"{exp_path}/{sub}"
    cnt = len(os.listdir(d)) if os.path.isdir(d) else 0
    print(f"{sub}: {cnt} file")
    if cnt == 0:
        raise RuntimeError(f"{sub} kosong. Baca error di atas.")

In [ ]:
#@title 6. Train Feature Index (opsional, tapi membuat suara lebih mirip)
import os, traceback
import numpy as np
import faiss
from sklearn.cluster import MiniBatchKMeans
os.chdir(now_dir)

fea_dim = 256 if version == "v1" else 768
feature_dir = f"{exp_path}/3_feature{fea_dim}"
files = sorted(os.listdir(feature_dir))
if not files:
    raise RuntimeError("Jalankan cell 5 dulu (feature kosong).")

big_npy = np.concatenate([np.load(f"{feature_dir}/{n}") for n in files], 0)
idx = np.arange(big_npy.shape[0])
np.random.shuffle(idx)
big_npy = big_npy[idx]

if big_npy.shape[0] > 2e5:
    print(f"Kmeans {big_npy.shape[0]} vektor -> 10k center")
    try:
        big_npy = MiniBatchKMeans(
            n_clusters=10000, verbose=True,
            batch_size=256 * (os.cpu_count() or 1),
            compute_labels=False, init="random",
        ).fit(big_npy).cluster_centers_
    except Exception:
        traceback.print_exc()

np.save(f"{exp_path}/total_fea.npy", big_npy)
n_ivf = min(int(16 * np.sqrt(big_npy.shape[0])), big_npy.shape[0] // 39)
print(big_npy.shape, n_ivf)

index = faiss.index_factory(fea_dim, f"IVF{n_ivf},Flat")
index_ivf = faiss.extract_index_ivf(index)
index_ivf.nprobe = 1
print("training")
index.train(big_npy)
faiss.write_index(index, f"{exp_path}/trained_IVF{n_ivf}_Flat_nprobe_{index_ivf.nprobe}_{exp_dir}_{version}.index")

print("adding")
for i in range(0, big_npy.shape[0], 8192):
    index.add(big_npy[i:i + 8192])
added = f"{exp_path}/added_IVF{n_ivf}_Flat_nprobe_{index_ivf.nprobe}_{exp_dir}_{version}.index"
faiss.write_index(index, added)
print("Selesai:", added)

In [ ]:
#@title 7. (Opsional) Restore checkpoint dari Drive
# Pakai HANYA kalau sesi ter-reset di tengah training dan kamu mau melanjutkan.
# Jalankan cell 3, 4, 5 dulu, baru cell ini, lalu cell 8.
import os, glob, shutil
backup_dir = f"/content/drive/MyDrive/backup_logs/{exp_dir}"
found = glob.glob(f"{backup_dir}/[GD]_*.pth")
if not found:
    print("Tidak ada backup di", backup_dir)
else:
    for f in found + [f"{backup_dir}/config.json"]:
        if os.path.exists(f):
            shutil.copy2(f, exp_path)
            print("restore:", os.path.basename(f))

In [ ]:
#@title 8. Train Model
import os, json, pathlib, shutil, glob, threading, time
from random import shuffle
from subprocess import Popen, PIPE, STDOUT

backup_to_drive = True #@param {type:"boolean"}

os.chdir(now_dir)
fea_dim = 256 if version == "v1" else 768
gt_wavs_dir = f"{exp_path}/0_gt_wavs"
feature_dir = f"{exp_path}/3_feature{fea_dim}"
f0_dir = f"{exp_path}/2a_f0"
f0nsf_dir = f"{exp_path}/2b-f0nsf"
spk_id = 0

def stems(d):
    return {n.split(".")[0] for n in os.listdir(d)}

names = stems(gt_wavs_dir) & stems(feature_dir) & stems(f0_dir) & stems(f0nsf_dir)
if not names:
    raise RuntimeError("Tidak ada data lengkap. Ulangi cell 4 dan 5.")

# --- filelist ---
opt = [
    f"{gt_wavs_dir}/{n}.wav|{feature_dir}/{n}.npy|{f0_dir}/{n}.wav.npy|{f0nsf_dir}/{n}.wav.npy|{spk_id}"
    for n in names
]
for _ in range(2):
    opt.append(
        f"{now_dir}/logs/mute/0_gt_wavs/mute{ksample_rate}.wav|{now_dir}/logs/mute/3_feature{fea_dim}/mute.npy"
        f"|{now_dir}/logs/mute/2a_f0/mute.wav.npy|{now_dir}/logs/mute/2b-f0nsf/mute.wav.npy|{spk_id}"
    )
shuffle(opt)
with open(f"{exp_path}/filelist.txt", "w") as f:
    f.write("\n".join(opt))
print("Filelist:", len(opt), "baris")

# --- config ---
if version == "v1" or ksample_rate == "40k":
    config_path = f"configs/v1/{ksample_rate}.json"
else:
    config_path = f"configs/v2/{ksample_rate}.json"
config_save_path = f"{exp_path}/config.json"
if not pathlib.Path(config_save_path).exists():
    with open(config_path) as cf:
        cfg = json.load(cf)
    with open(config_save_path, "w", encoding="utf-8") as f:
        json.dump(cfg, f, ensure_ascii=False, indent=4, sort_keys=True)
        f.write("\n")

# --- pretrained ---
pdir = "pretrained" if version == "v1" else "pretrained_v2"
G_path = f"assets/{pdir}/f0G{ksample_rate}.pth"
D_path = f"assets/{pdir}/f0D{ksample_rate}.pth"

# --- cek file ---
os.makedirs(f"{now_dir}/assets/weights", exist_ok=True)
bad = False
for p in [G_path, D_path,
          f"logs/mute/0_gt_wavs/mute{ksample_rate}.wav",
          f"logs/mute/3_feature{fea_dim}/mute.npy"]:
    ok = os.path.exists(f"{now_dir}/{p}")
    bad = bad or not ok
    print(("OK      " if ok else "HILANG  ") + p)
if bad:
    raise RuntimeError("Ada file HILANG, perbaiki dulu (lihat daftar di atas).")

# --- backup otomatis ke Drive tiap 10 menit ---
backup_dir = f"/content/drive/MyDrive/backup_logs/{exp_dir}"
stop = threading.Event()

def backup_once():
    os.makedirs(backup_dir, exist_ok=True)
    for f in glob.glob(f"{exp_path}/[GD]_*.pth") + [f"{exp_path}/config.json"]:
        if not os.path.exists(f) or time.time() - os.path.getmtime(f) < 60:
            continue  # lewati file yang baru/sedang ditulis
        dst = os.path.join(backup_dir, os.path.basename(f))
        shutil.copy2(f, dst + ".tmp")
        os.replace(dst + ".tmp", dst)

def backup_loop():
    while not stop.wait(600):
        try:
            backup_once()
        except Exception as e:
            print("backup gagal:", e)

if backup_to_drive:
    threading.Thread(target=backup_loop, daemon=True).start()

# --- jalankan training ---
cmd = (
    f'python -m train.train -e "{exp_dir}" -sr {ksample_rate} -f0 1 -bs {batch_size} -g 0 '
    f'-te {epoch} -se {save_frequency} -pg {G_path} -pd {D_path} -l 1 '
    f'-c {1 if cache_gpu else 0} -sw 0 -v {version}'
)
print(cmd)
p = Popen(cmd, shell=True, cwd=now_dir, stdout=PIPE, stderr=STDOUT, bufsize=1, universal_newlines=True)
for line in p.stdout:
    print(line.rstrip())
p.wait()
stop.set()

if backup_to_drive:
    try:
        backup_once()
    except Exception as e:
        print("backup akhir gagal:", e)

final = f"{now_dir}/assets/weights/{exp_dir}.pth"
if p.returncode == 0 and os.path.exists(final):
    print(f"\nSELESAI. Model: {final}")
else:
    print(f"\nTraining berhenti (kode {p.returncode}) atau model final belum ada. Baca log di atas.")

In [ ]:
#@title 9. Simpan hasil ke Drive
import os, glob, shutil, zipfile
out = f"/content/drive/MyDrive/Model/{exp_dir}"
os.makedirs(out, exist_ok=True)

files = [f"{now_dir}/assets/weights/{exp_dir}.pth"]
files += glob.glob(f"{exp_path}/added_*.index")
files += [f"{exp_path}/total_fea.npy"]

saved = []
for f in files:
    if os.path.exists(f):
        shutil.copy2(f, out)
        saved.append(f)
        print("OK     ", os.path.basename(f))
    else:
        print("TIDAK ADA", os.path.basename(f))

with zipfile.ZipFile(f"{out}/{exp_dir}.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in saved:
        z.write(f, os.path.basename(f))

print("\nTersimpan di:", out)
print("Pemakaian: .pth -> assets/weights/ | added_*.index -> assets/indices/")